# Text-to-SVG V2: QLoRA Fine-Tuning on Qwen2.5-Coder-1.5B

**NYU Deep Learning Spring 2026 — Kaggle Competition**

**Environment:** Google Colab Pro with A100 GPU + Google Drive

### Setup Instructions
1. Upload `train.csv` and `test.csv` to your Google Drive under `MyDrive/svg-competition/`
2. Set Runtime → Change runtime type → **GPU** (A100 if available)
3. Run all cells in order
4. Adapter weights will be saved to `MyDrive/svg-competition/svg-lora-adapter/`

### V2 Improvements over V1 (scored 11.23)
- Clean single training run (no interrupted restarts)
- Short system prompt + `<svg` prefill for inference (fixes echo bug)
- LoRA merge before inference (faster generation)
- Lenient SVG extraction with force-close for incomplete outputs
- Regex-based canvas fix for broken XML (avoids `""` double-quote bug)
- External dataset supplement (5K samples from deepseek-svg)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/svg-competition'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project directory: {PROJECT_DIR}')
print(f'Contents: {os.listdir(PROJECT_DIR)}')

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl cairosvg lxml pandas

In [ ]:
import os, re, time, random, json
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset, concatenate_datasets

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Torch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/svg-competition'

CONFIG = {
    # ── Model ──
    'model_name': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    'max_seq_length': 1536,

    # ── LoRA ──
    'lora_r': 32,
    'lora_alpha': 64,
    'lora_dropout': 0.05,

    # ── Training ──
    'learning_rate': 2e-4,
    'num_train_epochs': 2,
    'per_device_train_batch_size': 12,
    'gradient_accumulation_steps': 1,
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'max_grad_norm': 0.3,

    # ── Logging & Saving ──
    'logging_steps': 25,
    'save_steps': 500,
    'eval_steps': 500,
    'save_total_limit': 3,
    'output_dir': '/content/svg-lora-checkpoints',

    # ── Data ──
    'train_csv': f'{PROJECT_DIR}/train.csv',
    'test_csv': f'{PROJECT_DIR}/test.csv',
    'eval_fraction': 0.02,
    'max_svg_len': 8000,
    'min_svg_len': 80,
    'external_data_cap': 5000,

    # ── Adapter Output ──
    'adapter_save_dir': f'{PROJECT_DIR}/svg-lora-adapter-v2',
}

for key in ['train_csv', 'test_csv']:
    exists = os.path.exists(CONFIG[key])
    print(f'{key}: {"FOUND" if exists else "NOT FOUND ⚠️"}')
print(json.dumps(CONFIG, indent=2))

## 3. Load & Clean Data (Competition + External)

In [ ]:
ALLOWED_TAGS = {
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline',
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask',
    'linearGradient', 'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter'
}

SVG_REGEX = re.compile(r'<svg[\s\S]*?</svg>', flags=re.IGNORECASE)

def validate_svg(svg_text):
    if not svg_text or not isinstance(svg_text, str):
        return False, 'empty'
    if not svg_text.strip().startswith('<svg'):
        return False, 'no_svg_root'
    if len(svg_text) > 8000:
        return False, 'too_long'
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False, 'parse_error'
    path_count = 0
    for elem in root.iter():
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        if tag not in ALLOWED_TAGS:
            return False, f'disallowed_tag:{tag}'
        if tag == 'path':
            path_count += 1
    if path_count > 256:
        return False, 'too_many_paths'
    return True, 'ok'

def normalize_svg(svg_text):
    return re.sub(r'\s+', ' ', svg_text).strip()

def clean_dataframe(df, source_name=''):
    valid_rows = []
    reject_reasons = Counter()
    for _, row in df.iterrows():
        svg = str(row['svg']).strip()
        prompt = str(row['prompt']).strip()
        if len(svg) < CONFIG['min_svg_len']:
            reject_reasons['too_short'] += 1; continue
        if len(svg) > CONFIG['max_svg_len']:
            reject_reasons['filtered_long'] += 1; continue
        if not prompt or len(prompt) < 5:
            reject_reasons['bad_prompt'] += 1; continue
        is_valid, reason = validate_svg(svg)
        if not is_valid:
            reject_reasons[reason] += 1; continue
        valid_rows.append({'prompt': prompt, 'svg': normalize_svg(svg)})
    print(f'  {source_name}: {len(valid_rows)} clean / {len(df)} raw')
    for reason, count in reject_reasons.most_common(3):
        print(f'    rejected {reason}: {count}')
    return valid_rows

In [ ]:
# ── Load competition training data ──
print('Loading competition train.csv...')
df_comp = pd.read_csv(CONFIG['train_csv'])
comp_rows = clean_dataframe(df_comp, 'competition')

# ── Load external dataset (capped at 5K to stay in time budget) ──
print('\nLoading external dataset: thesantatitan/deepseek-svg-dataset...')
ext_rows = []
try:
    ds_ext = load_dataset('thesantatitan/deepseek-svg-dataset', split='train')
    # Find the prompt and SVG columns
    ext_prompt_fields = ['prompt', 'instruction', 'input']
    ext_svg_fields = ['completion', 'output', 'svg']

    temp_rows = []
    for ex in ds_ext:
        prompt = ''
        for f in ext_prompt_fields:
            if f in ex and ex[f] and str(ex[f]).strip():
                prompt = str(ex[f]).strip(); break
        svg = ''
        for f in ext_svg_fields:
            if f in ex and ex[f] and str(ex[f]).strip().startswith('<svg'):
                svg = str(ex[f]).strip(); break
        if prompt and svg:
            temp_rows.append({'prompt': prompt, 'svg': svg})

    temp_df = pd.DataFrame(temp_rows)
    if len(temp_df) > CONFIG['external_data_cap']:
        temp_df = temp_df.sample(CONFIG['external_data_cap'], random_state=SEED)
    ext_rows = clean_dataframe(temp_df, 'deepseek-svg')
except Exception as e:
    print(f'  Skipping external data: {e}')

# ── Combine ──
all_rows = comp_rows + ext_rows
random.shuffle(all_rows)
print(f'\nTotal training samples: {len(all_rows)} ({len(comp_rows)} competition + {len(ext_rows)} external)')

In [ ]:
# ── Train/val split ──
n_eval = max(100, int(len(all_rows) * CONFIG['eval_fraction']))
eval_rows = all_rows[:n_eval]
train_rows = all_rows[n_eval:]

train_dataset = Dataset.from_list(train_rows)
eval_dataset = Dataset.from_list(eval_rows)
print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

## 4. Format for SFT

Using a **short system prompt** — we found the long prompt caused the model to echo instructions during inference.

In [ ]:
TRAIN_SYSTEM_PROMPT = 'Generate SVG code.'

def format_chat(example):
    text = (
        '<|im_start|>system\n'
        f'{TRAIN_SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f"{example['prompt']}<|im_end|>\n"
        '<|im_start|>assistant\n'
        f"{example['svg']}<|im_end|>"
    )
    return {'text': text}

train_formatted = train_dataset.map(format_chat, remove_columns=train_dataset.column_names)
eval_formatted = eval_dataset.map(format_chat, remove_columns=eval_dataset.column_names)

print('Sample (first 500 chars):')
print(train_formatted[0]['text'][:500])

## 5. Load Model + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG['logging_steps'],
    eval_strategy='steps',
    eval_steps=CONFIG['eval_steps'],
    save_strategy='steps',
    save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    gradient_checkpointing=True,
    seed=SEED,
    dataloader_pin_memory=True,
    max_length=CONFIG['max_seq_length'],
    packing=False,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=eval_formatted,
    args=sft_config,
)

eff_batch = CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']
total_steps = (len(train_formatted) // eff_batch) * CONFIG['num_train_epochs']
print(f'Effective batch: {eff_batch} | Total steps: ~{total_steps}')

In [ ]:
t0 = time.time()
train_result = trainer.train()
elapsed = (time.time() - t0) / 60
print(f'\nTraining complete in {elapsed:.1f} minutes')
print(f'Final train loss: {train_result.training_loss:.4f}')

## 7. Save Adapter + Prepare for Inference

In [ ]:
# Save adapter to Drive
adapter_dir = CONFIG['adapter_save_dir']
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Adapter saved to: {adapter_dir}')

# Save training summary
summary = {
    'model': CONFIG['model_name'], 'lora_r': CONFIG['lora_r'],
    'epochs': CONFIG['num_train_epochs'], 'batch': eff_batch,
    'lr': CONFIG['learning_rate'], 'train_samples': len(train_formatted),
    'final_loss': train_result.training_loss, 'time_min': elapsed,
    'gpu': torch.cuda.get_device_name(0), 'seed': SEED,
}
with open(f'{PROJECT_DIR}/training_summary_v2.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Summary: {summary}')

In [ ]:
# ── Merge LoRA into base model for faster inference ──
model = model.merge_and_unload()
model.eval()
print('LoRA merged — ready for inference')

## 8. Inference — Generate Submission

Key fixes from V1:
- Short system prompt (`Generate SVG code.`) — prevents echo bug
- `<svg` prefill — forces model to start generating SVG immediately
- `repetition_penalty=1.4` — breaks repetition loops
- Force-close incomplete SVGs instead of falling back
- Regex-based canvas fix — avoids `ns0:` prefix and `""` double-quote bugs

In [ ]:
def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ''

def fallback_svg(prompt):
    colors = ['red', 'blue', 'green', 'yellow', 'orange', 'purple', 'black', 'white', 'pink', 'brown', 'gray']
    fill = 'gray'
    for c in colors:
        if c in prompt.lower():
            fill = c
            break
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        f'<rect width="256" height="256" fill="white"/>'
        f'<circle cx="128" cy="128" r="64" fill="{fill}"/>'
        '</svg>'
    )

def generate_svg_fast(prompt):
    chat_text = (
        '<|im_start|>system\n'
        'Generate SVG code.<|im_end|>\n'
        '<|im_start|>user\n'
        f'{prompt}<|im_end|>\n'
        '<|im_start|>assistant\n'
        '<svg'
    )
    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=768,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.4,
        )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)

    # Extract assistant output
    assistant_part = decoded.split('<|im_start|>assistant\n')[-1]
    for token in ['<|im_end|>', '<|im_start|>', '<|endoftext|>']:
        assistant_part = assistant_part.split(token)[0]

    # Try complete SVG first
    svg = extract_svg(assistant_part)

    # Force-close incomplete SVGs
    if not svg and '<svg' in assistant_part:
        start = assistant_part.index('<svg')
        svg_partial = assistant_part[start:].rstrip()
        last_self_close = svg_partial.rfind('/>')
        last_end_tag = svg_partial.rfind('</')
        if last_end_tag != -1:
            try:
                end_pos = svg_partial.index('>', last_end_tag) + 1
                svg_partial = svg_partial[:end_pos]
            except ValueError:
                if last_self_close != -1:
                    svg_partial = svg_partial[:last_self_close + 2]
        elif last_self_close != -1:
            svg_partial = svg_partial[:last_self_close + 2]
        if '</svg>' not in svg_partial:
            svg_partial += '</svg>'
        svg = svg_partial

    if svg:
        # Try XML-based canvas fix
        ET.register_namespace('', 'http://www.w3.org/2000/svg')
        try:
            root = ET.fromstring(svg)
            root.set('xmlns', 'http://www.w3.org/2000/svg')
            root.set('width', '256')
            root.set('height', '256')
            if 'viewBox' not in root.attrib:
                root.set('viewBox', '0 0 256 256')
            svg = ET.tostring(root, encoding='unicode')
            svg = svg.replace('ns0:', '').replace(':ns0', '')
        except ET.ParseError:
            # Regex-based fix for broken XML
            svg = re.sub(r'width="[^"]*"', 'width="256"', svg, count=1)
            svg = re.sub(r'height="[^"]*"', 'height="256"', svg, count=1)
            if 'viewBox' not in svg:
                svg = svg.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
            if 'xmlns' not in svg:
                svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)

        # Clean double-quotes bug
        svg = svg.replace('""', '"')

        if len(svg) <= 8000 and svg.strip().startswith('<svg') and '</svg>' in svg:
            return svg

    return fallback_svg(prompt)

# Quick test
for i in range(5):
    t1 = time.time()
    svg = generate_svg_fast(test_df.iloc[i]['prompt'] if 'test_df' in dir() else 'a red circle')
    print(f'[{i}] {time.time()-t1:.1f}s | len={len(svg)} | fallback={len(svg)<190}')

In [ ]:
# ── Load test prompts ──
test_df = pd.read_csv(CONFIG['test_csv'])
print(f'Test prompts: {len(test_df)} rows')

In [ ]:
# ── Generate all 1000 SVGs ──
rows = []
fallback_count = 0
t0 = time.time()

for idx, row in test_df.iterrows():
    prompt = str(row['prompt']).strip()
    t1 = time.time()
    svg = generate_svg_fast(prompt)
    gen_time = time.time() - t1

    is_fallback = len(svg) < 190
    if is_fallback:
        fallback_count += 1

    rows.append({'id': row['id'], 'svg': svg})
    print(f'  [{idx+1}/{len(test_df)}] {gen_time:.1f}s | len={len(svg)} | fallback={is_fallback} | total_fb={fallback_count}')

elapsed_total = (time.time() - t0) / 60
print(f'\nDone! {len(rows)} SVGs in {elapsed_total:.1f} min')
print(f'Fallbacks: {fallback_count}/{len(rows)} ({100*fallback_count/len(rows):.1f}%)')

In [ ]:
# ── Save and download submission ──
sub_df = pd.DataFrame(rows)

# Final double-quote cleanup
sub_df['svg'] = sub_df['svg'].str.replace('""', '"', regex=False)

svg_lengths = sub_df['svg'].str.len()
print(f'SVG lengths — mean: {svg_lengths.mean():.0f}, max: {svg_lengths.max():.0f}, over 8K: {(svg_lengths>8000).sum()}')
print(f'Rows: {len(sub_df)}')

SUBMISSION_PATH = f'{PROJECT_DIR}/submission_v2.csv'
sub_df.to_csv(SUBMISSION_PATH, index=False)
print(f'Saved to: {SUBMISSION_PATH}')

from google.colab import files
files.download(SUBMISSION_PATH)

In [ ]:
import re

sub_df = pd.read_csv(SUBMISSION_PATH)

def fix_duplicate_attrs(svg):
    # Remove duplicate xmlns
    svg = svg.replace('xmlns="http://www.w3.org/2000/svg" xmlns="http://www.w3.org/2000/svg"',
                       'xmlns="http://www.w3.org/2000/svg"')
    # Remove duplicate width/height
    svg = re.sub(r'(width="256")\s+width="256"', r'\1', svg)
    svg = re.sub(r'(height="256")\s+height="256"', r'\1', svg)
    # Fix double quotes
    svg = svg.replace('""', '"')
    return svg

sub_df['svg'] = sub_df['svg'].apply(fix_duplicate_attrs)

# Verify no duplicates remain
issues = 0
for idx, row in sub_df.iterrows():
    svg = row['svg']
    if svg.count('xmlns=') > 1:
        issues += 1
        print(f'Row {idx} still has duplicate xmlns')
    if svg.count('width="256"') > 1:
        issues += 1
        print(f'Row {idx} still has duplicate width')

print(f'\nIssues remaining: {issues}')
print(f'Sample fixed SVG:')
print(sub_df.iloc[1]['svg'][:300])

# Save fixed version
FIXED_PATH = f'{PROJECT_DIR}/submission_v2_fixed.csv'
sub_df.to_csv(FIXED_PATH, index=False)
print(f'\nSaved to: {FIXED_PATH}')

from google.colab import files
files.download(FIXED_PATH)

## AI Tooling Disclosure

- **Claude (Anthropic)**: Coding assistance, debugging.